<a href="https://colab.research.google.com/github/shaify-cloud/ai-mentor-portfolio/blob/main/Day9_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.2 MB/s eta 0:00:00


In [2]:
import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [5]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test the tool directly
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

February 25, 2026 - TCS will conduct the hiring round using the TCS iON platform rather than the TCS Community platform, which TCS has utilized in the past. Yes, students from 2024 can apply in this drive. January 20, 2026 - Tata Consultancy Services (TCS), India’s most trusted IT services company, has launched its TCS Hiring 2026 drive, creating a massive wave of job opportunities for freshers an


In [6]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_9943/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [7]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

# Print every message in the conversation
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': 'f53a083b-00d9-4d18-82a6-63d014ba651f', 'type': 'tool_call'}]

[2] ToolMessage
    Content: The TCS Lateral Drive 2026 is a golden opportunity for IT professionals with 2-4 years of experience in technologies like Java, Python, ReactJS, Node.js, Selenium, and Informatica ETL, who are eager to accelerate their careers with a global leader and contribute to groundbreaking AI projects. And al

[3] AIMessage
    Content: [{'type': 'text', 'text': 'I couldn\'t find a specific "hiring quota" for TCS in 2026. However, I found information about several hiring initiatives for 2026, including:\n\n*   **TCS Lateral Drive 2026:** For IT professionals with 2-4 years of experience in technologies like Java, Python, ReactJS, N


In [8]:
# Pass a question that should fail the tool
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

# Watch how the agent recovers
for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    [{'type': 'text', 'text': 'I cannot directly access the content of a URL. I can perform a web search based on a query.', 'extras': {'signature': 'CpEDAQw51seFWXfM5ITZ2B2D0PiXcBD7KN5/eRGXaVZYPvLlEE2OlMj5pgCwZfXEeEmRFZZt1OaJFh+gkqExRTs3rKSX/+1j1O0LXixXrb4Uw+DchJ+eV94RhBsS0I+WP884AEgtluLBxmIbMbuu5AlHEN
